# Convert MeatLens `.h5` to ONNX

This notebook converts:
- `training_outputs/models/meatlens_mobilenetv3small_cross_rotation_fold1_seed42_cnn_only.h5`

to ONNX and provides a download link.


In [1]:
from pathlib import Path
import json
import zipfile

import tensorflow as tf
from IPython.display import display, FileLink

print("TensorFlow:", tf.__version__)


TensorFlow: 2.10.0


## Install ONNX Conversion Dependencies (Run once per environment)
If `tf2onnx` and `onnx` are not yet installed in your notebook kernel, run the next cell.


In [2]:
# Uncomment and run if needed:
# %pip install -U tf2onnx onnx


In [3]:
try:
    import tf2onnx
    import onnx
except Exception as e:
    raise RuntimeError(
        "Missing dependency. Install with: %pip install -U tf2onnx onnx\n"
        f"Original error: {e}"
    )

print("tf2onnx:", tf2onnx.__version__)
print("onnx:", onnx.__version__)


tf2onnx: 1.16.1
onnx: 1.16.1


In [4]:
PROJECT_ROOT = Path.cwd()
MODELS_DIR = PROJECT_ROOT / "training_outputs" / "models"

H5_PATH = MODELS_DIR / "meatlens_mobilenetv3small_cross_rotation_fold1_seed42_cnn_only.h5"
META_PATH = MODELS_DIR / "meatlens_mobilenetv3small_cross_rotation_fold1_seed42_cnn_only_metadata.json"
ONNX_PATH = MODELS_DIR / "meatlens_mobilenetv3small_cross_rotation_fold1_seed42_cnn_only.onnx"
ZIP_PATH = MODELS_DIR / "meatlens_mobilenetv3small_cross_rotation_fold1_seed42_cnn_only.onnx.zip"

if not H5_PATH.exists():
    raise FileNotFoundError(f"Missing model file: {H5_PATH}")

if META_PATH.exists():
    meta = json.loads(META_PATH.read_text(encoding="utf-8"))
else:
    meta = {
        "input_size": [224, 224],
        "backbone": "MobileNetV3Small",
    }

input_h, input_w = meta.get("input_size", [224, 224])
print("H5_PATH:", H5_PATH)
print("META_PATH:", META_PATH)
print("ONNX_PATH:", ONNX_PATH)
print("Input size:", (input_h, input_w))


H5_PATH: e:\Thesis Code\training_outputs\models\meatlens_mobilenetv3small_cross_rotation_fold1_seed42_cnn_only.h5
META_PATH: e:\Thesis Code\training_outputs\models\meatlens_mobilenetv3small_cross_rotation_fold1_seed42_cnn_only_metadata.json
ONNX_PATH: e:\Thesis Code\training_outputs\models\meatlens_mobilenetv3small_cross_rotation_fold1_seed42_cnn_only.onnx
Input size: (224, 224)


In [5]:
model = tf.keras.models.load_model(H5_PATH, compile=False)

input_signature = [
    tf.TensorSpec(
        shape=(None, int(input_h), int(input_w), 3),
        dtype=tf.float32,
        name="input"
    )
]

_onnx_model, _external_tensor_storage = tf2onnx.convert.from_keras(
    model,
    input_signature=input_signature,
    opset=13,
    output_path=str(ONNX_PATH),
)

print("Conversion complete:", ONNX_PATH)


Conversion complete: e:\Thesis Code\training_outputs\models\meatlens_mobilenetv3small_cross_rotation_fold1_seed42_cnn_only.onnx


In [6]:
onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)

print("ONNX model is valid.")
print("ONNX file size (MB):", round(ONNX_PATH.stat().st_size / (1024 * 1024), 3))


ONNX model is valid.
ONNX file size (MB): 3.883


In [7]:
# Optional ZIP for easier download
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(ONNX_PATH, arcname=ONNX_PATH.name)

print("Created ZIP:", ZIP_PATH)
print("ZIP file size (MB):", round(ZIP_PATH.stat().st_size / (1024 * 1024), 3))


Created ZIP: e:\Thesis Code\training_outputs\models\meatlens_mobilenetv3small_cross_rotation_fold1_seed42_cnn_only.onnx.zip
ZIP file size (MB): 3.554


In [8]:
print("Download links:")
display(FileLink(str(ONNX_PATH)))
display(FileLink(str(ZIP_PATH)))


Download links:


e:\Thesis Code\training_outputs\models\meatlens_mobilenetv3small_cross_rotation_fold1_seed42_cnn_only.onnx

e:\Thesis Code\training_outputs\models\meatlens_mobilenetv3small_cross_rotation_fold1_seed42_cnn_only.onnx.zip